In [ ]:
# TODO: 
# ICU Info
# Critical Care Info

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Paths -- edit these for your environment.
RAW_DIR     = "/PATH/TO/raw"              # raw MGB extracts (edw/, clarity/, Datamart/)
DATA_DIR    = "/PATH/TO/data"             # project data root; lookup tables live here
WORK_DIR    = f"{DATA_DIR}/20250604"      # identified intermediate outputs -- keep on the secure server
DEID_DIR    = f"{WORK_DIR}/deidentified"  # de-identified outputs and ID / date-shift maps
RELEASE_DIR = "/PATH/TO/release"          # de-identified, date-shifted release files

In [ ]:
episode = pd.read_csv(f'{RAW_DIR}/edw/encounters/MGB_encounters_20240715.csv')
micro = pd.read_csv(f'{RAW_DIR}/Datamart/AllMGB_micro_20150525-20250201.csv')
mrn_map = pd.read_csv(f'{RAW_DIR}/edw/MRN_map/EDW_MRN_map_20240713.csv')
microbiology_cultures_cohort = pd.read_csv(f'{WORK_DIR}/microbiology_cultures_cohort.csv')

In [ ]:
episode = episode.rename(columns = {'PatientID':'patientID',
                                    'PatientEncounterID':'patientEncounterID',
                                    'visit_type':'adtPatientClassification'})

## Step 1: Extract ER and ICU Information from Episode Table
episode['hosp_ward_ER'] = np.where(episode['adtPatientClassification'] == 'Emergency', 1, 0)

## TODO:
episode['hosp_ward_ICU'] = np.nan
episode['hosp_ward_Critical_Care'] = np.nan

temp_er_info_adt = episode[['patientID','patientEncounterID','hosp_ward_ER','hosp_ward_ICU','hosp_ward_Critical_Care']]
temp_er_info_adt = temp_er_info_adt.rename(columns = {'patientID':'anon_id',
                                                      'patientEncounterID':'pat_enc_csn_id_coded'})

adt_er_info_cohort = microbiology_cultures_cohort[['anon_id','pat_enc_csn_id_coded','order_proc_id_coded']].merge(temp_er_info_adt,
                                                                                                                  on = ['anon_id','pat_enc_csn_id_coded'],
                                                                                                                  how = 'left')
adt_er_info_cohort = adt_er_info_cohort.drop_duplicates()

adt_er_info_cohort

In [ ]:
## Step 2: Extract ER and IP/OP Information from micro Table
micro['hosp_ward_ER_order_proc'] = np.where(micro['loc_cat'] == 'ER', 1, 0)
micro['hosp_ward_IP'] = np.where(micro['loc_cat'] == 'IP', 1, 0)
micro['hosp_ward_OP'] = np.where(micro['loc_cat'] == 'OP', 1, 0)

## Drop all columns not needed downstream — reduces memory before the merge and range join
micro = micro[['hospital', 'MRN', 'site_clean', 'accession', 'coll_datetime',
               'hosp_ward_ER_order_proc', 'hosp_ward_IP', 'hosp_ward_OP']]

## Filter to the 3 site types that appear in the cohort — reduces the range join significantly
micro = micro[micro['site_clean'].isin(['URINE', 'RESPIRATORY_TRACT', 'BLOOD'])]

### Map MRN to patientID
mrn_map = mrn_map.rename(columns={'PatientID':'patientID',
                                  'MRN':'mrn'})

micro = micro.merge(mrn_map[['patientID','hospital','mrn']],
                    left_on = ['hospital','MRN'],
                    right_on = ['hospital','mrn'])

## Prefix accession with hospital to match order_proc_id_coded in cohort
micro['accession'] = micro['hospital'] + '_' + micro['accession'].astype(str)

## Further filter to only accessions present in the cohort — reduces micro to exactly the rows needed
cohort_accessions = set(microbiology_cultures_cohort['order_proc_id_coded'])
micro = micro[micro['accession'].isin(cohort_accessions)]

### Get patientEncounterID via range join: culture collection time must fall
### within the encounter's admission–discharge window.
### Bug fix: old code merged on admit_date (date only), duplicating cultures
### for patients with two encounters starting on the same calendar date.
micro['coll_datetime'] = pd.to_datetime(micro['coll_datetime'])
episode['hospitalAdmitDTS'] = pd.to_datetime(episode['hospitalAdmitDTS'])
episode['hospitalDischargeDTS'] = pd.to_datetime(episode['hospitalDischargeDTS'])

micro = micro.merge(
    episode[['patientID','patientEncounterID','adtPatientClassification',
             'hospitalAdmitDTS','hospitalDischargeDTS']],
    on='patientID',
    how='inner'
)

micro = micro[
    (micro['coll_datetime'] >= micro['hospitalAdmitDTS']) &
    (micro['coll_datetime'] <= micro['hospitalDischargeDTS'].fillna(pd.Timestamp('2099-12-31')))
].copy()

temp_er_info_order_proc = micro[["patientID","patientEncounterID","accession","coll_datetime",
                                 "hosp_ward_ER_order_proc","hosp_ward_IP","hosp_ward_OP"]]
temp_er_info_order_proc = temp_er_info_order_proc.rename(columns = {'patientID':'anon_id',
                                                                    'patientEncounterID':'pat_enc_csn_id_coded',
                                                                    'accession':'order_proc_id_coded',
                                                                    'coll_datetime':'order_time_jittered_utc',})

temp_er_info_order_proc = temp_er_info_order_proc.drop_duplicates()

combined_er_icu_cohort = adt_er_info_cohort.merge(temp_er_info_order_proc,
                                                  on = ['anon_id','pat_enc_csn_id_coded','order_proc_id_coded'],
                                                  how = 'left')
combined_er_icu_cohort

In [ ]:
## Step 3: Extract ICU stay based on transfer orders

In [ ]:
## Step 4: Create the Final Table

combined_er_icu_cohort = combined_er_icu_cohort.drop_duplicates()

final_table = combined_er_icu_cohort.copy()
final_table["hosp_ward_ER"] = final_table[["hosp_ward_ER", "hosp_ward_ER_order_proc"]].max(axis=1)
final_table["hosp_ward_ICU"] = final_table[["hosp_ward_ICU", "hosp_ward_Critical_Care"]].max(axis=1)

final_table = final_table[['anon_id','pat_enc_csn_id_coded','order_proc_id_coded','order_time_jittered_utc',
                           'hosp_ward_IP','hosp_ward_OP','hosp_ward_ER','hosp_ward_ICU']]

final_table

In [ ]:
final_table.to_csv(f"{WORK_DIR}/microbiology_cultures_ward_info.csv",
                   index = False)

## Deidentification

In [ ]:
# Load the saved mappings (assume you saved as CSVs before)
anon_map_df = pd.read_csv(f'{DEID_DIR}/anon_id_map.csv')
enc_map_df = pd.read_csv(f'{DEID_DIR}/enc_id_map.csv')
order_proc_map_df = pd.read_csv(f'{DEID_DIR}/order_proc_id_map.csv')

In [ ]:
# Convert to Dict
anon_id_map = dict(zip(anon_map_df['anon_id'], anon_map_df['anon_id_deid']))
enc_id_map = dict(zip(enc_map_df['pat_enc_csn_id_coded'], enc_map_df['pat_enc_csn_id_deid']))
order_proc_map = dict(zip(order_proc_map_df['order_proc_id_coded'], order_proc_map_df['order_proc_id_deid']))

# Apply the mappings
final_table_deid = final_table.copy()
final_table_deid['anon_id'] = final_table_deid['anon_id'].map(anon_id_map)
final_table_deid['pat_enc_csn_id_coded'] = final_table_deid['pat_enc_csn_id_coded'].map(enc_id_map)
final_table_deid['order_proc_id_coded'] = final_table_deid['order_proc_id_coded'].map(order_proc_map)

final_table_deid

In [ ]:
final_table_deid.to_csv(f"{DEID_DIR}/microbiology_cultures_ward_info.csv",
                        index = False)

## From EDW

In [ ]:
cohort_df = pd.read_csv(f"{RELEASE_DIR}/microbiology_cohort_deid_tj.csv")
cohort_df

In [ ]:
cohort_df['ordering_mode'].unique()

In [ ]:
mode_map = {
    "Inpatient": "IP",
    "Psych Inpatient": "IP",
    "Outpatient": "OP",
    "Emergency": "ER",
    "Observation": "ER",
    "ED Observation": "ER",
    "Urgent Care": "Urgent Care",
    "Day Surgery": "Day Surgery",
    "Surgery Admit": "Day Surgery",
}

cohort_df['ordering_mode_collapsed'] = cohort_df['ordering_mode'].map(mode_map)

# Indicator flags
cohort_df['hosp_ward_IP'] = (cohort_df['ordering_mode_collapsed'] == 'IP').astype(int)
cohort_df['hosp_ward_OP'] = (cohort_df['ordering_mode_collapsed'] == 'OP').astype(int)
cohort_df['hosp_ward_ER'] = (cohort_df['ordering_mode_collapsed'] == 'ER').astype(int)
cohort_df['hosp_ward_Urgent_Care'] = (cohort_df['ordering_mode_collapsed'] == 'Urgent Care').astype(int)
cohort_df['hosp_ward_Day_Surgery'] = (cohort_df['ordering_mode_collapsed'] == 'Day Surgery').astype(int)

final_table = cohort_df[['anon_id','pat_enc_csn_id_coded','order_proc_id_coded','order_time_jittered_utc_shifted',
                         'hosp_ward_IP','hosp_ward_OP','hosp_ward_ER','hosp_ward_Urgent_Care','hosp_ward_Day_Surgery']]

final_table = final_table.drop_duplicates()

final_table

In [ ]:
final_table.to_csv(f"{RELEASE_DIR}/dedup/ward_type_edw_deid_tj.csv",
                    index = False)